# W7 Citation 측정 — by_page=True 재빌드 + prompt fix 검증

D1 디버그 결과 Citation 0%의 원인은 인프라 결함이었음:
- `vectorstore.create_vectorstore(by_page=False)` (기본값) → `parse_pdf`가 문서 전체를 `page_num=1`로 처리
- C3 청크 258개 전부 `page=1` → LLM이 항상 `p.1`만 인용 → tolerance ±2로도 0%

수정:
1. C3 재임베딩 (`by_page=True`) → 338 docs, page 1~56 정상 분산
2. `run_rag_pipeline` 프롬프트에 `[제품_복잡도 p.페이지]` 출처 태그 주입
3. LLM이 `(출처: ... p.N)` 형식으로 인용하도록 지시

이 노트북은 41q에서 baseline RAG (Hybrid+Rerank, **MPS**) 돌려 Citation Accuracy만 측정.
RAGAS는 측정 안 함 (비용·시간 절감).


In [1]:
# Cell 1: setup
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from src.vectorstore import load_vectorstore
from src.retrieval import HybridRerankerRetriever, create_reranker
from src.evaluation import (
    load_golden_set,
    run_rag_pipeline,
    citation_accuracy,
    extract_citations,
)
from langchain_openai import ChatOpenAI

vs = load_vectorstore(Path('../data/chroma_db_c3'), collection_name='lg_manuals_c3')
print(f'Vectorstore: {vs._collection.count()} docs')

retriever = HybridRerankerRetriever(vs)
# MPS reranker 주입 (default CPU 대신)
retriever.reranker = create_reranker(retriever.rerank_config.model_name, device='mps')

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

questions = load_golden_set(Path('../data/eval/golden_set_v2.csv'))
print(f'Golden set: {len(questions)} questions')


/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Vectorstore: 338 docs


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13502.31it/s]


Golden set: 41 questions


In [2]:
# Cell 2: run 41q baseline RAG
import time

answers, latencies = [], []
t0 = time.time()
for i, q in enumerate(questions):
    s = time.time()
    ans, _ = run_rag_pipeline(q.question, retriever, llm, k=5)
    latencies.append(time.time() - s)
    answers.append(ans)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(questions)}  avg {sum(latencies)/len(latencies):.2f}s/q')

print(f'Total {time.time()-t0:.1f}s  avg {sum(latencies)/len(latencies):.2f}s/q')


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  10/41  avg 29.56s/q
  20/41  avg 30.23s/q
  30/41  avg 31.10s/q
  40/41  avg 30.13s/q
Total 1231.5s  avg 30.03s/q


In [4]:
# Cell 3: Citation Accuracy + per-question table
import pandas as pd

cit = citation_accuracy(questions, answers, page_tolerance=2)
print(f"Citation Accuracy (±2 pages): {cit['accuracy']:.3f}  (n={cit['n']})")
print('By q_type:')
for qt, stats in cit['by_q_type'].items():
    print(f"  {qt:15s} {stats['correct']}/{stats['total']} = {stats['accuracy']:.1%}")

rows = []
for q, a in zip(questions, answers):
    cites = extract_citations(a)
    rows.append({
        'id': q.id,
        'q_type': q.q_type,
        'modality': getattr(q, 'modality_label', ''),
        'expected': (q.reference_context or '')[:35],
        'cited': ', '.join(f'{m} p.{p}' for m, p in cites) or '(none)',
    })

df = pd.DataFrame(rows)
df


Citation Accuracy (±2 pages): 0.189  (n=37)
By q_type:
  factual         5/27 = 18.5%
  multi_hop       0/4 = 0.0%
  comparison      2/4 = 50.0%
  safety          0/2 = 0.0%


,id,q_type,modality,expected,cited
0,Q01,factual,text-only,waterpurifier_simple p.15 (정수 필터 교체,waterpurifier_simple p.18
1,Q02,factual,text-only,waterpurifier_complex p.8-9 (제어창 사용,waterpurifier_complex p.15
2,Q03,factual,text-only,waterpurifier_simple p.20 (문제 해결하기),waterpurifier_simple p.23
3,Q04,factual,text-only,waterpurifier_complex p.12 (출수구 살균,waterpurifier_complex p.15
4,Q05,factual,text-only,waterpurifier_complex p.10 (온수 기능을,waterpurifier_simple p.14
5,Q06,factual,text-only,airpurifier_complex p.18 (필터 교체하기),waterpurifier_simple p.18
6,Q07,factual,image-helpful,airpurifier_simple p.14 (필터 청소하기),airpurifier_complex p.37
7,Q08,factual,text-only,airpurifier_complex p.10 (사용하기),airpurifier_simple p.25
8,Q09,factual,image-helpful,airpurifier_complex p.16 (센서 청소하기),airpurifier_simple p.38
9,Q10,factual,text-only,airpurifier_simple p.8 (상태 표시부 알림),airpurifier_simple p.28


In [5]:
# Cell 4: model match vs page match — separate the two failure modes
# Citation 인프라가 정상이면 model match는 높아야 함 (retrieval이 적어도 같은 제품 청크를 가져옴)
# page match는 retrieval이 expected page와 가까운 청크를 가져오느냐에 달림 — 인프라 문제와 분리해서 봐야 의미 있음
from src.evaluation import _parse_reference_citation

scored = model_hits = page_hits = no_cite = 0
for q, a in zip(questions, answers):
    exp = _parse_reference_citation(q.reference_context) if q.reference_context else None
    if exp is None:
        continue
    scored += 1
    exp_model, exp_page = exp
    exp_cat = exp_model.split('_')[0]
    cites = extract_citations(a)
    if not cites:
        no_cite += 1
        continue
    for cite_model, cite_page in cites:
        cite_cat = cite_model.split('_')[0]
        if cite_cat == exp_cat:
            model_hits += 1
            if abs(cite_page - exp_page) <= 2:
                page_hits += 1
            break

print(f'Scored questions   : {scored}')
print(f'No citation at all : {no_cite}')
print(f'Model match (cat)  : {model_hits}/{scored} = {model_hits/scored:.1%}')
print(f'Page match (±2)    : {page_hits}/{scored} = {page_hits/scored:.1%}')


Scored questions   : 37
No citation at all : 1
Model match (cat)  : 34/37 = 91.9%
Page match (±2)    : 7/37 = 18.9%
